# SOHO ImageNet-R train-only diagnostic

This notebook replays the exact six historical class/projection seed pairs to diagnose why selected SOHO loses AIA against original FLY on ImageNet-R. It never extracts or reads held-out test features. The fixed-1000 SOHO row is a post-hoc causal diagnostic, not a selected method or paper result.


In [ ]:
# === Edit repository/path values only. Do not edit seeds or method settings. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/soho_imagenetr_gcv_train_cache'
OUTPUT_DIR = '/content/soho_imagenetr_gcv_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_PROTOCOL_SHA256 = '83e11107a531ec8be49f2e32f9e40151e78ba621777894ae3baf7dc6858eb468'
EXPECTED_RUNNER_SHA256 = '38c4fa0d80367bcd31a25bd511d514ef1202032f2553b0210a430acb8241e06c'


In [ ]:
# Fresh clone, dependencies, GPU check and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
PROTOCOL = 'configs/soho_imagenetr_gcv_diagnostic.json'
RUNNER = 'tools/soho_imagenetr_gcv_diagnostic.py'
assert sha(PROTOCOL) == EXPECTED_PROTOCOL_SHA256, 'Protocol hash mismatch: pull the locked branch.'
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256, 'Runner hash mismatch: pull the locked branch.'
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Repository must start clean.'
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('TRAIN-ONLY DIAGNOSTIC SOURCE CHECK: PASS')


In [ ]:
# Download the verified frozen ViT checkpoint and processed ImageNet-R artifact.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOT = kagglehub.dataset_download('zaphat206/imagenet-r')
print('checkpoint:', CHECKPOINT_PATH)
print('ImageNet-R root:', DATASET_ROOT)


In [ ]:
# Verify the exact legacy processed split identity; do not extract any test feature.
AUDIT_PATH = '/content/imagenetr_soho_gcv_audit.json'
audit_run = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOT,'--output',AUDIT_PATH,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert audit_run.returncode == 2, 'Expected the locked legacy-overlap disclosure.'
audit = json.loads(Path(AUDIT_PATH).read_text())
assert audit['cross_split_duplicate_content_count'] == 19
assert audit['cross_split_conflicting_label_duplicate_count'] == 18
print('DATASET IDENTITY: PASS (legacy split disclosure retained)')


In [ ]:
# Synthetic correctness gate. Full output is printed if anything fails.
command = [sys.executable,'-m','pytest','-q','tests/test_soho_imagenetr_gcv_diagnostic.py','tests/test_cached_replay_baselines.py','tests/test_soho_selfcontained.py']
completed = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(completed.stdout, flush=True)
if completed.returncode != 0: raise RuntimeError(f'Correctness gate failed: {completed.returncode}')
print('SOHO IMAGENET-R DIAGNOSTIC CORRECTNESS GATE: PASS')


In [ ]:
# Extract TRAIN features only. The runner prints live task progress; no Drive storage is used.
cache = Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only',
      '--root',DATASET_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,
      '--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b',
      '--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_imagenetr_gcv',
      '--dataset','ImageNet-R','--model-name','vit_base_patch16_224','--data-augmentation','vit',
      '--seed','2025','--num-classes','200','--num-tasks','20','--device','cuda',
      '--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START - one progress line per task', flush=True)
    subprocess.run(command, check=True)
else:
    print('Using existing local TRAIN cache:', FEATURE_CACHE_DIR)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(), 'FAIL: held-out test cache is visible.'
metadata = json.loads((cache/'metadata.json').read_text())
assert metadata['train_shape'] == [23918,768] and metadata['finite'] is True
print('TRAIN CACHE PASS:', metadata['train_shape'], '| test.pt absent')


In [ ]:
# Run six exact historical seed pairs. Units are resume-safe inside this Colab runtime.
assert not (Path(FEATURE_CACHE_DIR)/'test.pt').exists()
command = [sys.executable,'-u',RUNNER,'--protocol',PROTOCOL,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda']
print('Starting ImageNet-R train-only diagnosis: 6 paired replicates x 20 tasks.', flush=True)
print('Each SOHO TASK line reports selected ridge, GCV/fixed validation AA and WTA support turnover.', flush=True)
started = time.time()
completed = subprocess.run(command)
assert completed.returncode == 0, 'Diagnostic failed; return the complete traceback without editing the protocol.'
assert Path(OUTPUT_DIR,'diagnostic_results.json').is_file()
print(f'DIAGNOSTIC PROCESS COMPLETE in {(time.time()-started)/60:.1f} minutes')


In [ ]:
# Compact numerical result and mechanism plot. This remains train-validation evidence only.
import pandas as pd
import matplotlib.pyplot as plt
payload = json.loads(Path(OUTPUT_DIR,'diagnostic_results.json').read_text())
summary = payload['summary']
rows = []
for method, metrics in summary['methods'].items():
    rows.append({'method':method,'AIA_mean':metrics['average_incremental_accuracy']['mean'],'AIA_std':metrics['average_incremental_accuracy']['sample_std'],'final_mean':metrics['final_accuracy']['mean'],'final_std':metrics['final_accuracy']['sample_std']})
display(pd.DataFrame(rows))
stage_rows = []
for i, rep in enumerate(payload['replicates']):
    for stage in rep['stage_diagnostics']:
        stage_rows.append({'replicate':i+1,**stage})
stage = pd.DataFrame(stage_rows)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for name, color in [('gcv_stage_accuracy','tab:red'),('fixed_stage_accuracy','tab:blue')]: axes[0].plot(stage.groupby('task')[name].mean(),label=name,color=color)
axes[0].set(xlabel='Task',ylabel='Validation seen-class AA (%)',title='SOHO Ridge diagnostic'); axes[0].legend()
valid = stage.dropna(subset=['probe_support_turnover'])
axes[1].scatter(valid['probe_support_turnover'],valid['gcv_stage_accuracy']-valid['fixed_stage_accuracy'],alpha=.65)
axes[1].axhline(0,color='black',lw=1); axes[1].set(xlabel='WTA support turnover',ylabel='GCV - fixed AA (pp)',title='Map instability vs Ridge effect')
plt.tight_layout(); plt.show()
print('STATUS:', summary['status'], '| held_out_test_authorized=', summary['held_out_test_authorized'])


In [ ]:
# Export compact evidence only; feature tensors are deliberately excluded.
archive_base = '/content/soho_imagenetr_gcv_diagnostic_train_only'
archive = shutil.make_archive(archive_base,'zip',root_dir=OUTPUT_DIR)
print('ZIP:', archive, 'bytes=', Path(archive).stat().st_size, 'sha256=', sha(archive))
from google.colab import files
files.download(archive)
print('STOP: return the ZIP for audit. Do not evaluate ImageNet-R test.')
